# 01 – Load & Plot a PiEEG Session

This notebook shows two ways to get EEG data:

1. **Load a previously recorded CSV** from `../recordings/`
2. **Stream live from the PiEEG server** via WebSocket and collect a buffer

Then we plot all 16 channels and zoom in on a window of interest.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SAMPLE_RATE = 250  # Hz

## Option A – Load from CSV

In [ ]:
import glob, os

csv_files = sorted(glob.glob("../recordings/pieeg_*.csv"))
print(f"Found {len(csv_files)} recording(s):")
for f in csv_files:
    print(f"  {os.path.basename(f)}")

In [ ]:
# Pick the most recent recording (or change the index)
df = pd.read_csv(csv_files[-1])

# Convert timestamp to seconds from start
df["time"] = df["timestamp"] - df["timestamp"].iloc[0]

ch_cols = [c for c in df.columns if c.startswith("ch")]
print(f"Channels: {len(ch_cols)}  |  Samples: {len(df)}  |  Duration: {df['time'].iloc[-1]:.1f}s")
df.head()

## Option B – Stream live from the server

Run `pieeg-server --mock` in a terminal first, then execute the cell below.
It collects 5 seconds of data (1 250 samples).

In [ ]:
import asyncio, json, websockets

SERVER_URI = "ws://localhost:1616"
COLLECT_SECONDS = 5

async def collect_stream(uri, seconds):
    rows = []
    async with websockets.connect(uri) as ws:
        welcome = json.loads(await ws.recv())
        print(f"Connected – {welcome['channels']} channels @ {welcome['sample_rate']} Hz")
        target = seconds * welcome["sample_rate"]
        while len(rows) < target:
            msg = json.loads(await ws.recv())
            if "channels" in msg:
                rows.append([msg["t"]] + msg["channels"])
    cols = ["timestamp"] + [f"ch{i+1}" for i in range(welcome["channels"])]
    return pd.DataFrame(rows, columns=cols)

# Uncomment the line below to stream live data:
# df = await collect_stream(SERVER_URI, COLLECT_SECONDS)
# df["time"] = df["timestamp"] - df["timestamp"].iloc[0]
# ch_cols = [c for c in df.columns if c.startswith("ch")]

## Plot all channels

In [ ]:
n_ch = len(ch_cols)
fig, axes = plt.subplots(n_ch, 1, figsize=(14, n_ch * 1.2), sharex=True)

for ax, ch in zip(axes, ch_cols):
    ax.plot(df["time"], df[ch], linewidth=0.4, color="steelblue")
    ax.set_ylabel(ch, rotation=0, labelpad=30, fontsize=9)
    ax.set_yticks([])

axes[-1].set_xlabel("Time (s)")
fig.suptitle("PiEEG – All Channels", fontsize=14)
plt.tight_layout()
plt.show()

## Zoom in on a 2-second window

In [ ]:
START_SEC = 1.0  # change this to explore
WINDOW_SEC = 2.0

mask = (df["time"] >= START_SEC) & (df["time"] < START_SEC + WINDOW_SEC)
snippet = df.loc[mask]

fig, axes = plt.subplots(4, 1, figsize=(14, 6), sharex=True)
for ax, ch in zip(axes, ch_cols[:4]):
    ax.plot(snippet["time"], snippet[ch], linewidth=0.6)
    ax.set_ylabel(ch, rotation=0, labelpad=30)

axes[-1].set_xlabel("Time (s)")
fig.suptitle(f"Channels 1–4  ({START_SEC}–{START_SEC+WINDOW_SEC} s)", fontsize=13)
plt.tight_layout()
plt.show()